In [ ]:
%pip install -r ../../requirements.txt

In [ ]:
from pathlib import Path
import sys

demos_dir = Path.cwd().parent
sys.path.insert(0, str(demos_dir)) # import path for sinter_decoders

import stim
import sinter
import numpy as np

# Decoder imports
import pymatching
import ldpc
from ldpc.sinter_decoders import SinterBpOsdDecoder, SinterLsdDecoder
from sinter_decoders.sinter_unionfind_decoder import SinterUnionFindDecoder

# Plotting
import matplotlib.pyplot as plt

## Surface Code

First we set up tasks for sinter

In [ ]:
distances = [3,5]
dep_error_rates = np.linspace(1e-5, 1e-2, 20)

# Source: adapted from https://github.com/quantumlib/Stim/blob/01f1aabd0fbc64d6943d42094a4bde96818399bd/doc/getting_started.ipynb

tasks = [
    sinter.Task(
        circuit=stim.Circuit.generated(
            "surface_code:rotated_memory_z",
            #"color_code:memory_xyz",
            rounds=d * 3,
            distance=d,
            before_round_data_depolarization=p, 
            after_clifford_depolarization=p, 
            before_measure_flip_probability=p, 
            after_reset_flip_probability=p
        ),
        json_metadata={'d': d, 'p': p},
    )
    for d in distances
    for p in dep_error_rates
]

Custom Decoders from ldpc imported and adapted. Then experiment set up and run using sinter

In [ ]:
custom_decoders = {
    "bposd": SinterBpOsdDecoder(
        max_iter=10,
        bp_method="ms",
        ms_scaling_factor=0.625,
        schedule="parallel",
        osd_method="osd0",
        osd_order=0,
    ),

    # Union-find decoder for sinter, adapted from https://github.com/quantumgizmos/ldpc/blob/main/src_python/ldpc/sinter_decoders/sinter_bposd_decoder.py
    "union_find": SinterUnionFindDecoder(
        uf_method='False'
    ),
}

collected_stats1 = sinter.collect(
    num_workers=20,
    tasks=tasks,
    decoders=['pymatching', 'bposd', 'union_find'],
    custom_decoders=custom_decoders,
    max_shots=1_000_000, # Decrease this to speed up, will be less accurate
    max_errors=10_000, # Decrease this to speed up, will be less accurate
)

Plot our error rates against eachother

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15,10))
sinter.plot_error_rate(
    ax=ax,
    stats=collected_stats1,
    x_func=lambda stats: stats.json_metadata['p'],
    group_func=lambda stats: f"{stats.decoder},d={stats.json_metadata['d']}",
)
ax.axline((0, 0), slope=1, label='p', linestyle='--', color='black')
ax.loglog()
ax.set_title("Surface Code Error Rates")
ax.set_xlabel("Phyical Error Rate")
ax.set_ylabel("Logical Error Rate per Shot")
ax.grid(which='major')
ax.grid(which='minor')
ax.legend()

Plot our runtimes against eachother per shot (decoding cycle)

In [ ]:
decoders1 = ["pymatching", "bposd", "union_find"]

for decoder in decoders1:
    runtimes_us = []

    for d in distances:
        matching_stats = [
            stat for stat in collected_stats1
            if stat.decoder == decoder
            and stat.json_metadata["d"] == d
        ]

        total_seconds = sum(stat.seconds for stat in matching_stats)
        total_shots = sum(stat.shots for stat in matching_stats)

        runtime_us = 1e6 * total_seconds / total_shots
        runtimes_us.append(runtime_us)

    plt.plot(distances, runtimes_us, marker='o', label=decoder)

plt.xlabel("Distance")
plt.ylabel("Runtime per cycle (μs)")
plt.title("Decoder runtime by distance, Surface code")
plt.xticks(distances)
plt.yscale("log")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## Colour Code
Do the same for colour code with LSD vs OSD decoders, set up tasks for sinter

In [ ]:
distances = [3, 5,7]
dep_error_rates = np.linspace(1e-5, 1e-2, 20)

# Source: adapted from https://github.com/quantumlib/Stim/blob/01f1aabd0fbc64d6943d42094a4bde96818399bd/doc/getting_started.ipynb

tasks = [
    sinter.Task(
        circuit=stim.Circuit.generated(
            #"surface_code:rotated_memory_z",
            "color_code:memory_xyz",
            rounds=d * 3,
            distance=d,
            before_round_data_depolarization=p, 
            after_clifford_depolarization=p, 
            before_measure_flip_probability=p, 
            after_reset_flip_probability=p
        ),
        json_metadata={'d': d, 'p': p},
    )
    for d in distances
    for p in dep_error_rates
]

Bring in custom decoders adapted from ldpc, in this experiment use bposd and bplsd

In [ ]:
# Decoders imported from ldpc github repo
custom_decoders = {
    "bposd": SinterBpOsdDecoder(
        max_iter=10,
        bp_method="ms",
        ms_scaling_factor=0.625,
        schedule="parallel",
        osd_method="osd0",
        osd_order=0,
    ),
    "bplsd": SinterLsdDecoder(
        max_iter=10,
        bp_method="ms",
        ms_scaling_factor=0.625,
        schedule="parallel",
        #omp_thread_count=10, Apparently they haven't implemented the parallel schedule yet for LSD...
        omp_thread_count=1,
        serial_schedule_order=None,
        lsd_order=0,
    ),
}

collected_stats = sinter.collect(
    num_workers=10,
    tasks=tasks,
    decoders=['pymatching','bposd','bplsd'],
    custom_decoders=custom_decoders,
    max_shots=1_000_000, # Decrease this to speed up, will be less accurate
    max_errors=10_000, # Decrease this to speed up, will be less accurate
)

Plot our error rates against eachother

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15,10))
sinter.plot_error_rate(
    ax=ax,
    stats=collected_stats,
    x_func=lambda stats: stats.json_metadata['p'],
    group_func=lambda stats: f"{stats.decoder},d={stats.json_metadata['d']}",
)
ax.axline((0, 0), slope=1, label='p', linestyle='--', color='black')
ax.loglog()
ax.set_title("Steane Code Error Rates")
ax.set_xlabel("Phyical Error Rate")
ax.set_ylabel("Logical Error Rate per Shot")
ax.grid(which='major')
ax.grid(which='minor')
ax.legend()

Plot our runtimes against eachother per shot (decoding cycle)

In [ ]:
decoders = ["pymatching", "bposd", "bplsd"]

for decoder in decoders:
    runtimes_us = []

    for d in distances:
        matching_stats = [
            stat for stat in collected_stats
            if stat.decoder == decoder
            and stat.json_metadata["d"] == d
        ]

        total_seconds = sum(stat.seconds for stat in matching_stats)
        total_shots = sum(stat.shots for stat in matching_stats)

        runtime_us = 1e6 * total_seconds / total_shots
        runtimes_us.append(runtime_us)

    plt.plot(distances, runtimes_us, marker='o', label=decoder)

plt.xlabel("Distance")
plt.ylabel("Runtime per cycle (μs)")
plt.title("Decoder runtime by distance, Steane code")
plt.xticks(distances)
plt.yscale("log")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()